### Purpose

웹 **설정** — 등록 승인자와 텔레그램 연결 배포 (재고 앱 "설정 · 텔레그램 연결"과 같은 방식).

| 대상 | 내용 |
|---|---|
| Lambda `dsglobaldbCreate` (갱신) | `GET/POST /gdbsettings`(Cognito): 승인자 추가·이름·빼기, 연결 링크·초대 링크, 초대 취소, 연결 끊기. `/internal/approvers`·`/internal/redeem`(봇 전용). 웹 승인은 승인자 누구나 |
| Lambda `dsglobaldbBot` (갱신) | 승인자 = 웹에서 연결한 텔레그램 + 비상용 `GDB_OWNER_TG`. `/start <코드>`로 연결 |
| 환경 변수 | core 에 `GDB_BOT_USERNAME`(연결 링크 `t.me/<봇>?start=`), `BOT_TABLE` |
| API Gateway | `/gdbsettings` GET·POST (Cognito 권한 부여자 `dsglobaldb-cognito`, `/regreview` 와 같은 템플릿) |
| 저장 | 봇 테이블 `dsglobaldb-bot`: `approver/<email>`, `tglink/<email>#<번호>`, `tgcode/<코드 해시>`(TTL) — 코드 원문은 저장하지 않음 |

- 슈퍼관리자는 그대로 `.env`의 `GDB_OWNER_EMAILS` → core 환경 변수 (웹에서 못 바꿈)
- 배포 전 테스트: `pytest AWS/Lambda/dsglobaldbCreate`, `pytest AWS/Lambda/dsglobaldbBot`
- **스테이지 배포 셀은 API 전체를 다시 배포한다**

In [ ]:
# Norton 등이 HTTPS를 가로채는 PC에서 SSL 오류 방지 - boto3 보다 먼저 적용해야 함
import truststore
truststore.inject_into_ssl()

import json
import time
import urllib.error
import urllib.request

import boto3

boto3.setup_default_session(profile_name='default', region_name='us-east-1')
from deploy_helpers import (deploy_lambda, ensure_resource_path, setup_lambda_method, setup_options, grant_invoke,
                            ensure_cognito_authorizer)

Settings

In [ ]:
ROLE_ARN = 'arn:aws:iam::199525800117:role/DSmapLambda'
ACCOUNT_ID = ROLE_ARN.split(':')[4]
API_ID = 'jyipsj28s9'
STAGE = 'dev'
BASE_URL = f'https://{API_ID}.execute-api.us-east-1.amazonaws.com/{STAGE}'
CORE_FUNCTION, CORE_SOURCE = 'dsglobaldbCreate', './dsglobaldbCreate/lambda_function.py'
BOT_FUNCTION, BOT_SOURCE = 'dsglobaldbBot', './dsglobaldbBot/lambda_function.py'
BOT_TABLE = 'dsglobaldb-bot'
TOKEN_PARAM = '/dsglobaldb/bot/token'
USER_POOL_ARN = f'arn:aws:cognito-idp:us-east-1:{ACCOUNT_ID}:userpool/us-east-1_W0IaBru9e'
AUTHORIZER_NAME = 'dsglobaldb-cognito'
# /regreview 와 같은 템플릿 — 권한 부여자가 확인한 로그인 사람(claims)을 Lambda 에 넘긴다
CLAIMS_TEMPLATE = '''{
  "body-json" : $input.json('$'),
  "params" : { "header" : {} },
  "context" : {
    "http-method" : "$context.httpMethod",
    "resource-path" : "$context.resourcePath",
    "claims-email" : "$util.escapeJavaScript($context.authorizer.claims.email)",
    "claims-name" : "$util.escapeJavaScript($context.authorizer.claims.name)",
    "claims-sub" : "$util.escapeJavaScript($context.authorizer.claims.sub)"
  }
}'''

token = boto3.client('ssm').get_parameter(Name=TOKEN_PARAM, WithDecryption=True)['Parameter']['Value']
with urllib.request.urlopen(f'https://api.telegram.org/bot{token}/getMe') as r:
    BOT_USERNAME = json.load(r)['result']['username']
print('봇:', '@' + BOT_USERNAME)

1. Lambda 2개 + core 환경 변수

In [ ]:
lam = boto3.client('lambda')

def set_env(function, values):
    lam.get_waiter('function_updated_v2').wait(FunctionName=function)
    current = lam.get_function_configuration(FunctionName=function).get('Environment', {}).get('Variables', {})
    lam.update_function_configuration(FunctionName=function, Environment={'Variables': {**current, **values}})
    lam.get_waiter('function_updated_v2').wait(FunctionName=function)

core_arn = deploy_lambda(CORE_FUNCTION, CORE_SOURCE, ROLE_ARN,
                         description='글로벌 DB 안전한 쓰기 (create · update/alias/history · 등록 요청 · 설정)', timeout=29, memory=256)
set_env(CORE_FUNCTION, {'GDB_BOT_USERNAME': BOT_USERNAME, 'BOT_TABLE': BOT_TABLE})
deploy_lambda(BOT_FUNCTION, BOT_SOURCE, ROLE_ARN, description='글로벌 DB 봇 — 등록 승인 카드 (텔레그램 webhook)',
              timeout=29, memory=256)
print('Lambda 갱신 끝')

2. `/gdbsettings` (Cognito) + 스테이지 배포 + 확인

In [ ]:
apigw = boto3.client('apigateway')
authorizer_id = ensure_cognito_authorizer(API_ID, AUTHORIZER_NAME, USER_POOL_ARN)
rid = ensure_resource_path(API_ID, 'gdbsettings')
for method in ('GET', 'POST'):
    setup_lambda_method(API_ID, rid, method, core_arn, CLAIMS_TEMPLATE, authorizer_id=authorizer_id)
setup_options(API_ID, rid, ['GET', 'POST'])
grant_invoke(CORE_FUNCTION, API_ID, ACCOUNT_ID, 'gdbsettings')
apigw.create_deployment(restApiId=API_ID, stageName=STAGE, description='web settings: /gdbsettings (Cognito)')

def status_of(path):
    # 새 경로는 1~2분 'Missing Authentication Token' 이 섞여 나온다 → 다시 시도
    for _ in range(15):
        try:
            with urllib.request.urlopen(BASE_URL + path) as r:
                return r.status
        except urllib.error.HTTPError as e:
            if b'Missing Authentication Token' not in e.read():
                return e.code
            time.sleep(8)
    raise RuntimeError(f'{path}: 경로가 반영되지 않았습니다')

print('로그인 없이 /gdbsettings:', status_of('/gdbsettings'))      # 401
res = lam.invoke(FunctionName=CORE_FUNCTION, Payload=json.dumps(
    {'context': {'http-method': 'POST', 'resource-path': '/internal/approvers'}, 'body-json': {}}).encode())
print('승인자 텔레그램 (웹에서 연결한 것):', json.loads(json.loads(res['Payload'].read())['body']))